# Clean and Transform Data

## Data Quality Checks

In [0]:
df=spark.read.format('csv')\
            .option('header','true')\
            .option('inferSchema','true')\
            .load('/Volumes/databricksrajanya/default/databricks_rajanya/BigMart Sales.csv')

In [0]:
display(df.limit(5))

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [0]:
from pyspark.sql.functions import col
df.select(col('Item_Fat_Content')).distinct().show()
df.select(col('Item_Type')).distinct().show()
df.select(col('Outlet_Size')).distinct().show()
df.select(col('Outlet_Location_Type')).distinct().show()
df.select(col('Outlet_Type')).distinct().show()

+----------------+
|Item_Fat_Content|
+----------------+
|         Low Fat|
|         Regular|
+----------------+

+--------------------+
|           Item_Type|
+--------------------+
|           Breakfast|
|         Hard Drinks|
|       Starchy Foods|
|              Canned|
|Fruits And Vegeta...|
|  Health And Hygiene|
|              Others|
|        Baking Goods|
|         Snack Foods|
|              Breads|
|             Seafood|
|               Dairy|
|         Soft Drinks|
|           Household|
|        Frozen Foods|
|                Meat|
+--------------------+

+-----------+
|Outlet_Size|
+-----------+
|       High|
|      Small|
|    Unknown|
|     Medium|
+-----------+

+--------------------+
|Outlet_Location_Type|
+--------------------+
|              Tier 3|
|              Tier 2|
|              Tier 1|
+--------------------+

+-----------------+
|      Outlet_Type|
+-----------------+
|Supermarket Type2|
|Supermarket Type1|
|    Grocery Store|
|Supermarket Type3|
+--------

In [0]:
print("Total Rows:", df.count())

print("Unique Rows:", df.dropDuplicates().count())

Total Rows: 8523
Unique Rows: 8523


##  Null Handling

### • Filling Item_Weight with average

In [0]:
from pyspark.sql.functions import avg
avg_weight=df.select(avg('Item_Weight')).collect()[0][0]
df=df.fillna(avg_weight,subset=['Item_Weight'])

In [0]:
df.filter(col('Item_Weight').isNull()).count()

0

### • Fill Outlet_Size with 'Unknown'

In [0]:
df=df.fillna('Unknown',subset=['Outlet_Size'])
df.filter(col('Outlet_Size').isNull()).count()

0

## Standardization of Categorical Values

### • Standardize Item_Fat_Content

In [0]:
from pyspark.sql.functions import when,col
df=df.withColumn('Item_Fat_Content',
                 when(col('Item_Fat_Content').isin('LF','low fat','Low Fat'),'Low Fat')
                 .when(col('Item_Fat_Content').isin('reg','Regular'),'Regular')
                 .otherwise(col("Item_Fat_Content"))
)
df.select('Item_Fat_Content').distinct().show()

+----------------+
|Item_Fat_Content|
+----------------+
|         Low Fat|
|         Regular|
+----------------+



### • Apply initcap() on Item_Type

In [0]:
from pyspark.sql.functions import initcap
df=df.withColumn('Item_Type',initcap(col('Item_Type')))
df.select('Item_Type').distinct().show()

+--------------------+
|           Item_Type|
+--------------------+
|           Breakfast|
|         Hard Drinks|
|       Starchy Foods|
|              Canned|
|Fruits And Vegeta...|
|  Health And Hygiene|
|              Others|
|        Baking Goods|
|         Snack Foods|
|              Breads|
|             Seafood|
|               Dairy|
|         Soft Drinks|
|           Household|
|        Frozen Foods|
|                Meat|
+--------------------+



## Derived Business Columns

### • GST = Item_MRP * 0.18

In [0]:
from pyspark.sql.functions import *
df=df.withColumn('GST', col("Item_MRP") * 0.18)
df.select('Item_MRP','GST').show(5)

+--------+------------------+
|Item_MRP|               GST|
+--------+------------------+
|249.8092|44.965655999999996|
| 48.2692| 8.688455999999999|
| 141.618|25.491239999999998|
| 182.095|           32.7771|
| 53.8614|          9.695052|
+--------+------------------+
only showing top 5 rows


### • Final_Price = Item_MRP + GST

In [0]:
from pyspark.sql.functions import col
df=df.withColumn('Final_Price',col("Item_MRP")+col("GST"))
df.select('Item_MRP','GST','Final_Price').show(5)

+--------+------------------+-----------------+
|Item_MRP|               GST|      Final_Price|
+--------+------------------+-----------------+
|249.8092|44.965655999999996|       294.774856|
| 48.2692| 8.688455999999999|        56.957656|
| 141.618|25.491239999999998|        167.10924|
| 182.095|           32.7771|         214.8721|
| 53.8614|          9.695052|63.55645200000001|
+--------+------------------+-----------------+
only showing top 5 rows


### Outlet age

In [0]:
df=df.withColumn('Outlet_Age',year(current_date())-col("Outlet_Establishment_Year"))
df.select('Outlet_Establishment_Year','Outlet_Age').show(5)


+-------------------------+----------+
|Outlet_Establishment_Year|Outlet_Age|
+-------------------------+----------+
|                     1999|        27|
|                     2009|        17|
|                     1999|        27|
|                     1998|        28|
|                     1987|        39|
+-------------------------+----------+
only showing top 5 rows


## ### • Assume 20% Profit Margin

In [0]:
df=df.withColumn('Estimated_Profit',col("Item_Outlet_Sales")*0.20)
df.select('Item_Outlet_Sales','Estimated_Profit').show(5)

+-----------------+------------------+
|Item_Outlet_Sales|  Estimated_Profit|
+-----------------+------------------+
|         3735.138|          747.0276|
|         443.4228|          88.68456|
|          2097.27|           419.454|
|           732.38|           146.476|
|         994.7052|198.94104000000002|
+-----------------+------------------+
only showing top 5 rows


### Profit Category

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "Profit_Category",
    when(col("Estimated_Profit") > 600, "High Profit")
    .when(col("Estimated_Profit") > 300, "Medium Profit")
    .otherwise("Low Profit")
)
df.select('Estimated_Profit','Profit_Category').show(5)

+------------------+---------------+
|  Estimated_Profit|Profit_Category|
+------------------+---------------+
|          747.0276|    High Profit|
|          88.68456|     Low Profit|
|           419.454|  Medium Profit|
|           146.476|     Low Profit|
|198.94104000000002|     Low Profit|
+------------------+---------------+
only showing top 5 rows


### 

### Mention Price_Band

In [0]:
df=df.withColumn('Price_Band',when(col("Item_MRP")>200,"Premium")
                 .otherwise("Budget"))
df.select('Item_MRP','Price_Band').show(5)

+--------+----------+
|Item_MRP|Price_Band|
+--------+----------+
|249.8092|   Premium|
| 48.2692|    Budget|
| 141.618|    Budget|
| 182.095|    Budget|
| 53.8614|    Budget|
+--------+----------+
only showing top 5 rows


Sale Category

In [0]:
df=df.withColumn('Sale_Category',when(col('Item_Outlet_Sales')>3000,"High Sale")
                 .when(col('Item_Outlet_Sales')>1500,"Medium Sale")
                 .otherwise("Low Sale"))

df.select('Item_Outlet_Sales','Sale_Category').show(5)


+-----------------+-------------+
|Item_Outlet_Sales|Sale_Category|
+-----------------+-------------+
|         3735.138|    High Sale|
|         443.4228|     Low Sale|
|          2097.27|  Medium Sale|
|           732.38|     Low Sale|
|         994.7052|     Low Sale|
+-----------------+-------------+
only showing top 5 rows


## Audit Columns
### • Load_Timestamp using current_timestamp()

In [0]:
df=df.withColumn('Load_Timestamp',current_timestamp())
df.select('Load_Timestamp').show(5)

+--------------------+
|      Load_Timestamp|
+--------------------+
|2026-06-02 11:09:...|
|2026-06-02 11:09:...|
|2026-06-02 11:09:...|
|2026-06-02 11:09:...|
|2026-06-02 11:09:...|
+--------------------+
only showing top 5 rows


### • Data_Source using lit('BigMart CSV')

In [0]:
df=df.withColumn('Data_Source',lit("BigMart CSV"))
df.select('Data_Source').show(5)

+-----------+
|Data_Source|
+-----------+
|BigMart CSV|
|BigMart CSV|
|BigMart CSV|
|BigMart CSV|
|BigMart CSV|
+-----------+
only showing top 5 rows


##  Window Functions

### Rank Products by Sales Within Each Outlet

In [0]:
from pyspark.sql.functions import rank, col
from pyspark.sql.window import Window
df=df.withColumn('Sales_Rank',rank().over(Window.partitionBy('Outlet_Identifier')\
                            .orderBy(col('Item_Outlet_Sales').desc())))

df.select('Outlet_Identifier','Item_Outlet_Sales','Sales_Rank')\
    .show(5000)                          

+-----------------+-----------------+----------+
|Outlet_Identifier|Item_Outlet_Sales|Sales_Rank|
+-----------------+-----------------+----------+
|           OUT010|        1775.6886|         1|
|           OUT010|        1575.2828|         2|
|           OUT010|        1524.0162|         3|
|           OUT010|        1482.0708|         4|
|           OUT010|        1342.2528|         5|
|           OUT010|         1314.955|         6|
|           OUT010|         1288.323|         7|
|           OUT010|         1281.665|         8|
|           OUT010|        1162.4868|         9|
|           OUT010|        1102.5648|        10|
|           OUT010|        1094.5752|        11|
|           OUT010|        1090.5804|        12|
|           OUT010|        1050.6324|        13|
|           OUT010|        1046.6376|        14|
|           OUT010|        1046.6376|        14|
|           OUT010|         1041.977|        16|
|           OUT010|        1034.6532|        17|
|           OUT010| 

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "Top_Seller",
    when(col("Sales_Rank") == 1, "Yes")
    .otherwise("No")
)
df.display()

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,GST,Final_Price,Outlet_Age,Estimated_Profit,Profit_Category,Price_Band,Sale_Category,Load_Timestamp,Data_Source,Sales_Rank,Top_Seller
NCK30,14.85,Low Fat,0.102065622,Household,254.2698,OUT010,1998,Unknown,Tier 3,Grocery Store,1775.6886,45.768564,300.038364,28,355.13772,Medium Profit,Premium,Medium Sale,2026-06-02T13:24:12.919Z,BigMart CSV,1,Yes
FDJ55,12.8,Regular,0.039385992,Meat,224.8404,OUT010,1998,Unknown,Tier 3,Grocery Store,1575.2828,40.471272,265.311672,28,315.05656,Medium Profit,Premium,Medium Sale,2026-06-02T13:24:12.919Z,BigMart CSV,2,No
FDV59,13.35,Low Fat,0.080387424,Breads,219.2166,OUT010,1998,Unknown,Tier 3,Grocery Store,1524.0162,39.458988,258.675588,28,304.80324,Medium Profit,Premium,Medium Sale,2026-06-02T13:24:12.919Z,BigMart CSV,3,No
NCX41,19.0,Low Fat,0.0,Health And Hygiene,211.0244,OUT010,1998,Unknown,Tier 3,Grocery Store,1482.0708,37.984392,249.00879200000003,28,296.41416,Low Profit,Premium,Low Sale,2026-06-02T13:24:12.919Z,BigMart CSV,4,No
FDM08,10.1,Regular,0.089688978,Fruits And Vegetables,225.5088,OUT010,1998,Unknown,Tier 3,Grocery Store,1342.2528,40.591584,266.100384,28,268.45056,Low Profit,Premium,Low Sale,2026-06-02T13:24:12.919Z,BigMart CSV,5,No
FDY02,8.945,Regular,0.146701312,Dairy,262.291,OUT010,1998,Unknown,Tier 3,Grocery Store,1314.955,47.212379999999996,309.50338,28,262.991,Low Profit,Premium,Low Sale,2026-06-02T13:24:12.919Z,BigMart CSV,6,No
FDK28,5.695,Low Fat,0.109784056,Frozen Foods,256.0646,OUT010,1998,Unknown,Tier 3,Grocery Store,1288.323,46.09162799999999,302.156228,28,257.6646,Low Profit,Premium,Low Sale,2026-06-02T13:24:12.919Z,BigMart CSV,7,No
FDP15,15.2,Low Fat,0.0,Meat,256.033,OUT010,1998,Unknown,Tier 3,Grocery Store,1281.665,46.08594,302.11894,28,256.333,Low Profit,Premium,Low Sale,2026-06-02T13:24:12.919Z,BigMart CSV,8,No
FDZ16,16.85,Regular,0.267565911,Frozen Foods,194.1478,OUT010,1998,Unknown,Tier 3,Grocery Store,1162.4868,34.946603999999994,229.094404,28,232.49736,Low Profit,Budget,Low Sale,2026-06-02T13:24:12.919Z,BigMart CSV,9,No
FDQ45,9.5,Regular,0.0,Snack Foods,182.3608,OUT010,1998,Unknown,Tier 3,Grocery Store,1102.5648,32.824944,215.185744,28,220.51296000000002,Low Profit,Budget,Low Sale,2026-06-02T13:24:12.919Z,BigMart CSV,10,No


##  Validation Checks

In [0]:
from pyspark.sql.functions import col
for c in df.columns:
    print(c,df.filter(col(c).isNull()).count())

Item_Identifier 0
Item_Weight 0
Item_Fat_Content 0
Item_Visibility 0
Item_Type 0
Item_MRP 0
Outlet_Identifier 0
Outlet_Establishment_Year 0
Outlet_Size 0
Outlet_Location_Type 0
Outlet_Type 0
Item_Outlet_Sales 0
GST 0
Final_Price 0
Outlet_Age 0
Estimated_Profit 0
Profit_Category 0
Price_Band 0
Sale_Category 0
Load_Timestamp 0
Data_Source 0
Sales_Rank 0
Top_Seller 0


In [0]:
df.select("Item_Fat_Content").distinct().show()

+----------------+
|Item_Fat_Content|
+----------------+
|         Low Fat|
|         Regular|
+----------------+



In [0]:
df.select("Item_Type").distinct().show()

+--------------------+
|           Item_Type|
+--------------------+
|           Breakfast|
|         Hard Drinks|
|       Starchy Foods|
|              Canned|
|Fruits And Vegeta...|
|  Health And Hygiene|
|              Others|
|        Baking Goods|
|         Snack Foods|
|              Breads|
|             Seafood|
|               Dairy|
|         Soft Drinks|
|           Household|
|        Frozen Foods|
|                Meat|
+--------------------+



In [0]:
df.select(
    "GST",
    "Final_Price",
    "Outlet_Age",
    "Price_Band",
    "Sale_Category"
).show(10)

+------------------+------------------+----------+----------+-------------+
|               GST|       Final_Price|Outlet_Age|Price_Band|Sale_Category|
+------------------+------------------+----------+----------+-------------+
|44.965655999999996|        294.774856|        27|   Premium|    High Sale|
| 8.688455999999999|         56.957656|        17|    Budget|     Low Sale|
|25.491239999999998|         167.10924|        27|    Budget|  Medium Sale|
|           32.7771|          214.8721|        28|    Budget|     Low Sale|
|          9.695052| 63.55645200000001|        39|    Budget|     Low Sale|
|          9.252144|         60.652944|        17|    Budget|     Low Sale|
|         10.378584|         68.037384|        39|    Budget|     Low Sale|
|         19.397196|127.15939600000002|        41|    Budget|    High Sale|
|         17.455068|        114.427668|        24|    Budget|     Low Sale|
|33.807852000000004|        221.629252|        19|    Budget|    High Sale|
+-----------

In [0]:
df.select(
    "Outlet_Identifier",
    "Item_Outlet_Sales",
    "Sales_Rank"
).orderBy(
    "Outlet_Identifier",
    "Sales_Rank"
).show(20)

+-----------------+-----------------+----------+
|Outlet_Identifier|Item_Outlet_Sales|Sales_Rank|
+-----------------+-----------------+----------+
|           OUT010|        1775.6886|         1|
|           OUT010|        1575.2828|         2|
|           OUT010|        1524.0162|         3|
|           OUT010|        1482.0708|         4|
|           OUT010|        1342.2528|         5|
|           OUT010|         1314.955|         6|
|           OUT010|         1288.323|         7|
|           OUT010|         1281.665|         8|
|           OUT010|        1162.4868|         9|
|           OUT010|        1102.5648|        10|
|           OUT010|        1094.5752|        11|
|           OUT010|        1090.5804|        12|
|           OUT010|        1050.6324|        13|
|           OUT010|        1046.6376|        14|
|           OUT010|        1046.6376|        14|
|           OUT010|         1041.977|        16|
|           OUT010|        1034.6532|        17|
|           OUT010| 

In [0]:
df.write.format('delta')\
    .mode("overwrite")\
    .saveAsTable("BigMart_silver")

spark.table("bigmart_silver").show(5)


+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+---------+------------------+----------+----------------+---------------+----------+-------------+--------------------+-----------+----------+----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|  Outlet_Type|Item_Outlet_Sales|      GST|       Final_Price|Outlet_Age|Estimated_Profit|Profit_Category|Price_Band|Sale_Category|      Load_Timestamp|Data_Source|Sales_Rank|Top_Seller|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+---------+------------------+----------+----------------+---------------+----------+-----